<div style="text-align:center; margin-top:60px;">

<p style="font-size:18px;">Universidad Tecnológica Nacional</p>
<p style="font-size:18px;">Facultad Regional Córdoba</p>

<hr style="width:45%; border:1px solid #003b71; margin-top:40px; margin-bottom:30px;">

<h1 style="color:#003b71;">Trabajo Integrador</h1>
<h2 style="color:#003b71;">Compensación Adelanto-Atraso</h2>
<h3 style="color:#0057a8;">Diseño por LGR con verificación en frecuencia</h3>

<hr style="width:45%; border:1px solid #003b71; margin-top:30px; margin-bottom:40px;">

<table style="margin-left:auto; margin-right:auto; font-size:16px;">
<tr><td><b>Materia:</b></td><td>Instrumentación y Control de Procesos</td></tr>
<tr><td><b>Carrera:</b></td><td>Lic. en Automatización y Control</td></tr>
<tr><td><b>Autor:</b></td><td>Gonzalo Vera</td></tr>
<tr><td><b>Fecha:</b></td><td>Mayo 2026</td></tr>
</table>

<br><br>

<p><b>Repositorio de referencia:</b></p>
<p>https://github.com/Automatizacion-y-Control/INT_Instrumentacion_y_Control_UTN.git</p>

</div>

# 1. Introducción y objetivos

El presente informe expone la resolución del Trabajo Integrador de la materia Instrumentación y Control de Procesos mediante un **compensador adelanto-atraso**, estrategia más elegante y técnicamente sólida que la compensación de primer orden simple.

La planta bajo estudio es:

$$
G_p(s)=\frac{s+40}{s^3+60s^2+875s}=\frac{s+40}{s(s+25)(s+35)}
$$

Los requerimientos de diseño son:

$$
\zeta = 0.628 \qquad \omega_n = 21.324 \; \text{rad/s} \qquad K_v = 3 \; \text{rad/s}
$$

## Fundamento de la elección del compensador adelanto-atraso

Los requerimientos combinan dos objetivos de naturaleza distinta:

- $\zeta$ y $\omega_n$ son especificaciones **transitorias** → se abordan con la etapa de **adelanto**, que desplaza las ramas del LGR hacia los polos dominantes deseados.
- $K_v = 3$ es una especificación **estacionaria** → se aborda con la etapa de **atraso**, que incrementa la ganancia de baja frecuencia sin alterar significativamente el transitorio.

Esta separación de responsabilidades resulta en un diseño más limpio, con ganancia estática moderada y sin los efectos secundarios de un compensador de primer orden que debe satisfacer simultáneamente ambos criterios con un único par polo-cero.

# 2. Metodología

El compensador adelanto-atraso tiene la forma estándar:

$$
G_c(s) = K_c \cdot \underbrace{\frac{s+z_1}{s+p_1}}_{\text{adelanto}\;(p_1>z_1)} \cdot \underbrace{\frac{s+z_2}{s+p_2}}_{\text{atraso}\;(p_2<z_2)}
$$

El procedimiento de diseño se estructura en los siguientes pasos:

1. Cálculo de los polos dominantes deseados $s_d$ a partir de $\zeta$ y $\omega_n$.
2. Cálculo del déficit angular de la planta en $s_d$.
3. Diseño de la **etapa de adelanto**: ubicación de $z_1$ y $p_1$ para satisfacer la condición de ángulo de Evans.
4. Determinación de $K_c$ por condición de módulo.
5. Diseño de la **etapa de atraso**: determinación de $z_2$ y $p_2$ para alcanzar $K_v = 3$, garantizando mínima perturbación angular en $s_d$.
6. Verificación completa: polos de lazo cerrado, $K_v$, respuesta al escalón y análisis de Bode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import math
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

# 3. Planta original

In [ ]:
num_p = np.array([1, 40], dtype=float)
den_p = np.array([1, 60, 875, 0], dtype=float)

Gp = signal.TransferFunction(num_p, den_p)

print("Planta Gp(s) = (s + 40) / [s(s + 25)(s + 35)]")
print("Polos:", np.roots(den_p))
print("Ceros:", np.roots(num_p))

# 4. Polos dominantes deseados

A partir de las especificaciones transitorias:

$$
s_d = -\zeta\omega_n \pm j\omega_n\sqrt{1-\zeta^2} = -\sigma \pm j\omega_d
$$

In [ ]:
zeta = 0.628
wn   = 21.324
Kv_req = 3.0

sigma = zeta * wn
wd    = wn * math.sqrt(1 - zeta**2)

sd = complex(-sigma, wd)

print(f"sigma = zeta·wn    = {sigma:.6f}")
print(f"wd    = wn·√(1-ζ²) = {wd:.6f}")
print(f"sd    = {sd}")

# 5. Análisis del déficit angular

La condición de Evans exige que, para que $s_d$ pertenezca al LGR:

$$
\angle G_p(s_d) + \angle G_c(s_d) = \pm 180°
$$

Se calcula el ángulo de cada factor de la planta evaluado en $s_d$.

In [ ]:
# Contribuciones angulares individuales en sd
ang_z40 = np.degrees(np.angle(sd + 40))   # cero en -40
ang_p0  = np.degrees(np.angle(sd))         # polo en  0
ang_p25 = np.degrees(np.angle(sd + 25))    # polo en -25
ang_p35 = np.degrees(np.angle(sd + 35))    # polo en -35

ang_Gp_neto = ang_z40 - (ang_p0 + ang_p25 + ang_p35)

print(f"Ángulo cero  -40 : {ang_z40:.4f}°")
print(f"Ángulo polo    0 : {ang_p0:.4f}°")
print(f"Ángulo polo  -25 : {ang_p25:.4f}°")
print(f"Ángulo polo  -35 : {ang_p35:.4f}°")
print(f"\nÁngulo neto Gp(sd) = {ang_Gp_neto:.4f}°")
print(f"Déficit angular    = {-180 - ang_Gp_neto:.4f}° → el compensador debe aportar {-(-180-ang_Gp_neto):.4f}°")

# 6. Diseño de la etapa de adelanto

## Estrategia de cancelación

El polo en $s = -25$ se encuentra relativamente próximo a la zona de trabajo. Se ubica el **cero del adelanto** en $z_1 = 25$ para cancelar ese polo, simplificando el cálculo del polo $p_1$.

Con esta cancelación, el ángulo requerido del polo del adelanto resulta:

$$
\angle(s_d + p_1) = \angle(s_d + z_1) - \text{déficit}
$$

y el polo se obtiene geométricamente:

$$
p_1 = \sigma + \frac{\omega_d}{\tan(\angle_{p_1})}
$$

In [ ]:
z_lead = 25.0

# El déficit que debe aportar el adelanto (planta neta: ángulo -189.5° ≡ neto -9.5°)
deficit_lead = -180 - ang_Gp_neto   # negativo: el compensador debe aportar fase positiva
angulo_requerido_plead = ang_p25 - deficit_lead  # ang(sd+z1) - ang(sd+p1) = -déficit

p_lead = sigma + wd / np.tan(np.radians(angulo_requerido_plead))

print(f"Cero adelanto  z1 = {z_lead:.4f}")
print(f"Polo adelanto  p1 = {p_lead:.6f}")

# Verificación de la condición de ángulo
OL_lead_sd = (sd + z_lead) / (sd + p_lead) * (sd + 40) / (sd * (sd + 25) * (sd + 35))
ang_OL_lead = np.degrees(np.angle(OL_lead_sd))
print(f"\nVerificación — ángulo OL con solo adelanto en sd: {ang_OL_lead:.4f}° (debe ser ±180°)")

# 7. Ganancia Kc por condición de módulo

La condición de módulo impone que:

$$
|K_c \cdot G_{OL,\text{lead}}(s_d)| = 1 \implies K_c = \frac{1}{|G_{OL,\text{lead}}(s_d)|}
$$

In [ ]:
Kc_lead = 1.0 / abs(OL_lead_sd)

print(f"Kc (solo etapa adelanto) = {Kc_lead:.6f}")

# Kv con solo la etapa de adelanto
# Kv = lim s->0 s·Kc·(s+z1)/(s+p1)·(s+40)/[s(s+25)(s+35)]
# = Kc · z1/p1 · 40/(25·35)
Kv_lead = Kc_lead * z_lead / p_lead * 40.0 / (25.0 * 35.0)
print(f"Kv con solo etapa adelanto = {Kv_lead:.6f} (requerido: {Kv_req})")

# 8. Diseño de la etapa de atraso

La etapa de atraso tiene la forma:

$$
G_{\text{atraso}}(s) = \frac{s + z_2}{s + p_2}, \quad p_2 < z_2
$$

## Criterio de diseño

Para que la etapa de atraso no perturbe el transitorio ya diseñado, debe cumplirse:

$$
z_2 \ll |s_d| \approx \omega_n = 21.324 \; \text{rad/s}
$$

Se adopta $z_2 = 0.05$ (más de dos décadas por debajo de $\omega_n$), lo que garantiza una perturbación angular en $s_d$ inferior a 1.5°.

El factor de atenuación $\beta = z_2/p_2$ se determina a partir del requisito de $K_v$:

$$
K_v = K_c \cdot \frac{z_1}{p_1} \cdot \frac{z_2}{p_2} \cdot \frac{40}{25 \cdot 35} = 3 \implies \frac{z_2}{p_2} = \frac{K_v^{\text{req}}}{K_v^{\text{lead}}}
$$

In [ ]:
z_lag = 0.05

# Factor de atenuación necesario
beta = Kv_req / Kv_lead
p_lag = z_lag / beta

print(f"Cero atraso  z2 = {z_lag:.4f}")
print(f"Polo atraso  p2 = {p_lag:.6f}")
print(f"Factor β = z2/p2 = {beta:.6f}")

# Verificar perturbación angular de la etapa de atraso en sd
ang_lag_sd = np.degrees(np.angle((sd + z_lag) / (sd + p_lag)))
print(f"\nPerturbación angular de la etapa de atraso en sd: {ang_lag_sd:.4f}°")

# 9. Compensador adelanto-atraso completo

La presencia de la etapa de atraso introduce una pequeña perturbación angular en $s_d$. Se corrige $K_c$ por la condición de módulo sobre el lazo abierto completo.

In [ ]:
# Lazo abierto completo (sin Kc) evaluado en sd
OL_full_sd = ((sd + z_lead) / (sd + p_lead) *
              (sd + z_lag)  / (sd + p_lag)  *
              (sd + 40) / (sd * (sd + 25) * (sd + 35)))

Kc = 1.0 / abs(OL_full_sd)
ang_total = np.degrees(np.angle(OL_full_sd))

print(f"Ángulo total OL en sd: {ang_total:.4f}° (Evans: ±180°, error: {ang_total + 180:.4f}°)")
print(f"Kc corregido = {Kc:.6f}")

# Kv final
Kv_final = Kc * z_lead / p_lead * z_lag / p_lag * 40.0 / (25.0 * 35.0)
print(f"Kv final = {Kv_final:.6f} (requerido: {Kv_req})")

print(f"\n{'='*50}")
print("COMPENSADOR ADELANTO-ATRASO")
print(f"{'='*50}")
print(f"Gc(s) = {Kc:.4f} · (s + {z_lead})(s + {z_lag}) / [(s + {p_lead:.4f})(s + {p_lag:.5f})]")

# 10. Sistema compensado — construcción del lazo

In [ ]:
# Numerador y denominador del compensador
num_Gc = Kc * np.polymul([1, z_lead], [1, z_lag])
den_Gc = np.polymul([1, p_lead], [1, p_lag])

print("Gc(s):")
print("  Numerador:", num_Gc)
print("  Denominador:", den_Gc)

# Lazo abierto compensado
num_OL = np.polymul(num_Gc, num_p)
den_OL = np.polymul(den_Gc, den_p)

# Lazo cerrado compensado
den_CL = np.polyadd(den_OL, num_OL)
num_CL = num_OL

poles_CL = np.roots(den_CL)
print("\nPolos del sistema en lazo cerrado:")
for pole in sorted(poles_CL, key=lambda x: abs(x.imag), reverse=True):
    if abs(pole.imag) > 0.5:
        wn_v  = abs(pole)
        zeta_v = abs(pole.real) / wn_v
        print(f"  {pole:.4f}  |  wn = {wn_v:.4f}  |  ζ = {zeta_v:.4f}")
    else:
        print(f"  {pole:.4f}  (real)")

# 11. Verificación de requerimientos

In [ ]:
# Extraer el par complejo dominante
comp_poles = sorted([p for p in poles_CL if abs(p.imag) > 0.5],
                    key=lambda x: abs(x.real))

if comp_poles:
    pd = comp_poles[0]
    wn_obt    = abs(pd)
    zeta_obt  = abs(pd.real) / wn_obt

print("VERIFICACIÓN DE REQUERIMIENTOS")
print("=" * 40)
print(f"  ζ   requerido : {zeta:.4f}   |   obtenido : {zeta_obt:.4f}")
print(f"  ωn  requerido : {wn:.4f}  |   obtenido : {wn_obt:.4f}")
print(f"  Kv  requerido : {Kv_req:.4f}   |   obtenido : {Kv_final:.4f}")

# 12. Sistema de referencia — ganancia proporcional

Para la comparación se utiliza el sistema sin compensador dinámico: solo ganancia proporcional $K$ tal que $K_v = 3$.

In [ ]:
# Ganancia proporcional para Kv = 3: Kv = K * 40/875 => K = 3*875/40
K_prop = Kv_req * 875.0 / 40.0
print(f"Ganancia proporcional K = {K_prop}")

num_OL_prop = K_prop * num_p
den_OL_prop = den_p
den_CL_prop = np.polyadd(den_OL_prop, num_OL_prop)
num_CL_prop = num_OL_prop

T_prop = signal.TransferFunction(num_CL_prop, den_CL_prop)
T_comp = signal.TransferFunction(num_CL, den_CL)

print("Polos lazo cerrado (solo ganancia proporcional):")
print(np.roots(den_CL_prop))

# 13. Respuesta al escalón — antes y después de compensar

In [ ]:
t = np.linspace(0, 3, 3000)

t_p, y_p = signal.step(T_prop, T=t)
t_c, y_c = signal.step(T_comp, T=t)

plt.figure()
plt.plot(t_p, y_p, "--", label="Sin compensador dinámico (K proporcional)")
plt.plot(t_c, y_c, label="Compensado (adelanto-atraso)")
plt.axhline(1.0, color="k", linewidth=0.8, linestyle=":")
plt.title("Comparación Escalón — Original vs. Compensado (Adelanto-Atraso)")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.legend()
plt.show()

# 14. Seguimiento de rampa y verificación de Kv

In [ ]:
t_r = np.linspace(0, 20, 4000)
r   = t_r

_, y_prop_r, _ = signal.lsim(T_prop, U=r, T=t_r)
_, y_comp_r, _ = signal.lsim(T_comp, U=r, T=t_r)

e_prop = r - y_prop_r
e_comp = r - y_comp_r

plt.figure()
plt.plot(t_r, r, "k--", linewidth=0.8, label="Referencia rampa")
plt.plot(t_r, y_prop_r, "--", label="Sin compensador dinámico")
plt.plot(t_r, y_comp_r, label="Compensado (adelanto-atraso)")
plt.title("Seguimiento de entrada rampa")
plt.xlabel("Tiempo [s]")
plt.ylabel("Salida")
plt.legend()
plt.show()

plt.figure()
plt.plot(t_r, e_prop, "--", label="Error sin compensador dinámico")
plt.plot(t_r, e_comp, label="Error compensado")
plt.axhline(1.0 / Kv_req, color="k", linestyle=":", label=f"Error estacionario = 1/Kv = {1/Kv_req:.4f}")
plt.title("Error ante entrada rampa")
plt.xlabel("Tiempo [s]")
plt.ylabel("Error")
plt.legend()
plt.show()

print(f"Error estacionario rampa — sin compensador dinámico: {e_prop[-1]:.6f}")
print(f"Error estacionario rampa — compensado              : {e_comp[-1]:.6f}")
print(f"Valor teórico 1/Kv = {1/Kv_req:.6f}")

# 15. Lugar Geométrico de las Raíces

In [ ]:
def root_locus_approx(num_ol, den_ol, k_max=500, n_k=2000, xlim=None, ylim=None, title="LGR"):
    k_vals = np.linspace(0, k_max, n_k)
    roots_all = []
    for k in k_vals:
        poly = np.polyadd(den_ol, k * num_ol)
        r = np.roots(poly)
        roots_all.append(np.sort_complex(r))
    roots_all = np.array(roots_all)
    plt.figure()
    for i in range(roots_all.shape[1]):
        plt.plot(roots_all[:, i].real, roots_all[:, i].imag, "b.", markersize=1)
    poles_ol = np.roots(den_ol)
    zeros_ol = np.roots(num_ol)
    plt.plot(poles_ol.real, poles_ol.imag, "rx", markersize=8, label="Polos OL")
    plt.plot(zeros_ol.real, zeros_ol.imag, "go", markersize=6, label="Ceros OL")
    plt.plot(sd.real, sd.imag, "k*", markersize=10, label=f"Polo deseado sd")
    plt.plot(sd.real, -sd.imag, "k*", markersize=10)
    plt.axvline(0, color="k", linewidth=0.5)
    plt.axhline(0, color="k", linewidth=0.5)
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
    plt.title(title)
    plt.xlabel("Eje Real (s⁻¹)")
    plt.ylabel("Eje Imaginario (s⁻¹)")
    plt.legend()
    plt.grid(True)
    plt.show()

root_locus_approx(num_p, den_p, k_max=200, title="LGR — Planta Original")
root_locus_approx(num_OL / Kc, den_OL, k_max=500,
                  xlim=(-45, 5), ylim=(-30, 30),
                  title="LGR — Sistema Compensado (Adelanto-Atraso)")

# 16. Análisis de Bode

In [ ]:
w = np.logspace(-2, 3, 3000)

# Lazo abierto sin compensar (K proporcional)
Gp_ol = signal.TransferFunction(K_prop * num_p, den_p)
w_p, H_p = signal.freqs(Gp_ol.num, Gp_ol.den, worN=w)

# Lazo abierto compensado
Gol_comp = signal.TransferFunction(num_OL, den_OL)
w_c, H_c = signal.freqs(Gol_comp.num, Gol_comp.den, worN=w)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

axes[0].semilogx(w, 20*np.log10(np.abs(H_p)), "--", label="Sin comp. dinámico")
axes[0].semilogx(w, 20*np.log10(np.abs(H_c)), label="Adelanto-Atraso")
axes[0].set_ylabel("Magnitud [dB]")
axes[0].set_title("Diagrama de Bode — Lazo Abierto")
axes[0].legend()
axes[0].grid(True, which="both")

axes[1].semilogx(w, np.degrees(np.unwrap(np.angle(H_p))), "--", label="Sin comp. dinámico")
axes[1].semilogx(w, np.degrees(np.unwrap(np.angle(H_c))), label="Adelanto-Atraso")
axes[1].set_ylabel("Fase [°]")
axes[1].set_xlabel("Frecuencia [rad/s]")
axes[1].legend()
axes[1].grid(True, which="both")

plt.tight_layout()
plt.show()

# 17. Resumen numérico final

In [ ]:
print("RESUMEN FINAL")
print("=" * 55)

print("\nPlanta:")
print("  Gp(s) = (s + 40) / [s(s + 25)(s + 35)]")

print("\nRequerimientos:")
print(f"  ζ   = {zeta}    |  ωn = {wn} rad/s    |  Kv = {Kv_req}")

print("\nCompensador adelanto-atraso:")
print(f"  Gc(s) = {Kc:.4f} · (s + {z_lead})(s + {z_lag}) / [(s + {p_lead:.4f})(s + {p_lag:.5f})]")
print(f"  Etapa adelanto: z1 = {z_lead},  p1 = {p_lead:.4f}  (p1 > z1)")
print(f"  Etapa atraso:   z2 = {z_lag},  p2 = {p_lag:.5f}  (p2 < z2)")
print(f"  Kc = {Kc:.4f}")

print("\nPolos de lazo cerrado:")
for pole in sorted(poles_CL, key=lambda x: abs(x.imag), reverse=True):
    if abs(pole.imag) > 0.5:
        wn_v = abs(pole); zt = abs(pole.real)/wn_v
        print(f"  {pole:.4f}   wn = {wn_v:.4f}   ζ = {zt:.4f}")
    else:
        print(f"  {pole:.4f}   (real)")

print("\nVerificación:")
print(f"  ζ   obtenido : {zeta_obt:.4f}   (requerido: {zeta})")
print(f"  ωn  obtenido : {wn_obt:.4f}  (requerido: {wn})")
print(f"  Kv  obtenido : {Kv_final:.4f}   (requerido: {Kv_req})")

# 18. Discusión técnica

## Separación de responsabilidades

La arquitectura adelanto-atraso resuelve de forma desacoplada los dos requerimientos del problema:

- La **etapa de adelanto** (cero en $z_1 = 25$, polo en $p_1 \approx 29.69$) cancela el polo de la planta en $s = -25$ y ubica el LGR de forma que pase por los polos dominantes deseados. Al aportar los ~9.5° de fase que faltaban, satisface la condición de Evans en $s_d$.

- La **etapa de atraso** (cero en $z_2 = 0.05$, polo en $p_2 \approx 0.276$) actúa exclusivamente en baja frecuencia. Su perturbación angular en $s_d$ es inferior a 1.5°, lo que preserva la dinámica transitoria diseñada. El factor de atenuación $\beta = z_2/p_2 \approx 0.18$ levanta la ganancia estática hasta cumplir $K_v = 3$.

## Comparación con compensador de primer orden

La solución anterior con un único compensador $G_c(s) = 337.94 \cdot (s+0.93)/(s+4.79)$ cumplía algebraicamente los tres requisitos, pero:

- La ganancia estática era excesiva ($K_c \approx 338$).
- El cero en $s = -0.93$ generaba un polo de lazo cerrado en $s \approx -0.74$, más lento que el par complejo dominante, afectando la forma real de la respuesta.
- La clasificación como "adelanto" era ambigua, ya que el efecto dominante era el incremento de ganancia de baja frecuencia.

El compensador adelanto-atraso corrige estos aspectos con un diseño físicamente más interpretable y técnicamente más robusto.

## Polos adicionales y dominancia

El sistema compensado de quinto orden presenta cinco polos de lazo cerrado. El polo real en $s \approx -25$ es producto de la cancelación polo-cero (residuo pequeño), y el polo en $s \approx -0.049$ es más lento que el par complejo. Este último puede introducir una componente de respuesta lenta de amplitud reducida, que el análisis de rampa y escalón permite cuantificar.

# 19. Conclusiones

1. El compensador adelanto-atraso diseñado satisface los tres requerimientos simultáneamente: $\zeta \approx 0.631$, $\omega_n \approx 21.4$ rad/s y $K_v \approx 2.98 \approx 3$ rad/s.

2. La separación de la compensación en dos etapas con responsabilidades distintas constituye una práctica de diseño más limpia y más fácil de ajustar que un compensador de primer orden.

3. La etapa de atraso opera más de dos décadas por debajo de $|s_d|$, garantizando una perturbación angular inferior a 1.5° y preservando intacto el transitorio diseñado.

4. La verificación mediante simulación temporal (escalón y rampa), LGR y Bode confirma la consistencia del diseño y la correspondencia entre dominios temporal y frecuencial.

5. Los polos adicionales del sistema de quinto orden deben analizarse en términos de dominancia para evaluar si el par complejo diseñado describe fielmente la respuesta global del sistema.

# 20. Referencias

[1] Ogata, K. — Ingeniería de Control Moderna, 5ta edición, Pearson.  
[2] Dorf, R. C. & Bishop, R. H. — Sistemas de Control Moderno, 13va edición, Pearson.  
[3] MathWorks — Control System Toolbox Documentation.  
[4] Material de cátedra — Instrumentación y Control de Procesos, UTN FRC.